# 11.01 Anthropic Prompt Caching Middleware

Claude 모델 전용 **프롬프트 캐시** 미들웨어 `AnthropicPromptCachingMiddleware` 를 사용한다.
긴 system prompt, tool 정의, 초기 대화 컨텍스트를 서버 측에 5분~1시간 캐시해 **입력 토큰 비용과 지연을 크게 낮춘다**.

## 학습 목표

- `AnthropicPromptCachingMiddleware`의 4개 파라미터(`type`, `ttl`, `min_messages_to_cache`, `unsupported_model_behavior`)를 이해한다
- 캐시 효과를 확인할 수 있는 `usage` 메타데이터를 읽는다 (`cache_creation_input_tokens`, `cache_read_input_tokens`)
- 1시간(`1h`) TTL beta와 5분 기본값의 선택 기준을 안다
- 비(非) Anthropic 모델에 이 미들웨어를 적용할 때의 동작(`warn` / `ignore` / `raise`)을 구분한다

## 언제 쓰나

- **긴 system prompt**(예: 수천 토큰의 기업 정책)를 매 턴 전송하는 에이전트
- **큰 tool 정의 목록**(예: 20+ 도구의 JSON schema)을 반복 전송하는 설정
- **RAG 컨텍스트 재사용**: 같은 문서 묶음을 여러 질의에 걸쳐 사용
- **멀티턴 대화**에서 앞부분 컨텍스트가 안정적인 경우

## 11.01.1 환경 설정

필요 패키지: `langchain`, `langchain-anthropic`. `.env`에 `ANTHROPIC_API_KEY`가 있어야 한다.

In [ ]:
# !pip install -U langchain langchain-anthropic

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)
model.invoke("ping").content[:50]

## 11.01.2 미들웨어 기본 사용

`AnthropicPromptCachingMiddleware()` 를 `create_agent(..., middleware=[...])`에 넣으면, LangChain이 알아서 **system prompt · tool 정의 · 마지막 메시지** 위치에 `cache_control: {"type": "ephemeral"}` 마커를 부착한다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_anthropic.middleware import AnthropicPromptCachingMiddleware

@tool
def get_policy(topic: str) -> str:
    """특정 주제에 대한 사내 정책을 반환한다."""
    return f"[policy/{topic}] ..."

# 일부러 긴 시스템 프롬프트 (1000+ 토큰 정도)로 캐시 효과를 눈에 보이게 만든다
LONG_SYSTEM = (
    "당신은 사내 IT 헬프데스크 에이전트입니다. 아래 정책을 항상 준수합니다.\n\n"
    + ("\n".join(f"- 정책 {i}: 사용자 요청을 처리할 때 로그를 남기고, 민감정보를 필터링합니다." for i in range(1, 60)))
)

agent = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[get_policy],
    system_prompt=LONG_SYSTEM,
    middleware=[AnthropicPromptCachingMiddleware()],
)

## 11.01.3 캐시 적중 확인

연속으로 두 번 호출하면 **첫 호출**은 `cache_creation_input_tokens`가 커지고, **두 번째 호출**은 `cache_read_input_tokens`로 이동한다.
두 번째 호출의 입력 토큰 단가가 약 90% 할인된다.

In [ ]:
def show_usage(result):
    last = result["messages"][-1]
    usage = getattr(last, "usage_metadata", None) or {}
    print("입력/출력 토큰:", usage.get("input_tokens"), "/", usage.get("output_tokens"))
    details = usage.get("input_token_details", {})
    print("cache_creation:", details.get("cache_creation"))
    print("cache_read    :", details.get("cache_read"))

r1 = agent.invoke({"messages": [{"role": "user", "content": "VPN 정책 알려줘"}]})
print("--- 1회차 ---"); show_usage(r1)

r2 = agent.invoke({"messages": [{"role": "user", "content": "MDM 정책도 알려줘"}]})
print("\n--- 2회차 ---"); show_usage(r2)

## 11.01.4 파라미터 전체

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `type` | `"ephemeral"` | 현재 Anthropic이 지원하는 유일한 타입 |
| `ttl` | `"5m"` | 캐시 유지 시간. `"5m"` 또는 `"1h"`. 1시간은 Anthropic beta 기능 |
| `min_messages_to_cache` | `0` | 대화 메시지 수가 이 값 이상일 때만 cache_control 부착 (짧은 대화에 불필요한 캐시 방지) |
| `unsupported_model_behavior` | `"warn"` | 비 Anthropic 모델에 적용 시 `"warn"` / `"ignore"` / `"raise"` |

### 1시간 TTL 예

In [ ]:
agent_1h = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[get_policy],
    system_prompt=LONG_SYSTEM,
    middleware=[
        AnthropicPromptCachingMiddleware(
            ttl="1h",                     # 긴 세션/배치 작업용
            min_messages_to_cache=2,      # 아주 짧은 대화는 캐시하지 않음
            unsupported_model_behavior="warn",
        )
    ],
)

## 11.01.5 비 Anthropic 모델에서의 동작

이 미들웨어는 내부적으로 `cache_control` 블록을 주입하므로, 다른 공급자 모델에서는 무시되거나 에러가 된다.
교체 가능한 파이프라인을 설계할 때는 `unsupported_model_behavior` 를 명시하라.

In [ ]:
from langchain_openai import ChatOpenAI

# warn: 경고 출력 후 cache_control 없이 호출 (권장 기본값)
# ignore: 조용히 비활성
# raise: 예외 (테스트 환경에서 오설정을 빨리 드러내고 싶을 때)
mw_warn = AnthropicPromptCachingMiddleware(unsupported_model_behavior="warn")

agent_mixed = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[get_policy],
    middleware=[mw_warn],
)

agent_mixed.invoke({"messages": [{"role": "user", "content": "테스트"}]})

## 11.01.6 Bedrock에서의 Claude

AWS Bedrock 경유로 Claude를 호출할 때는 `BedrockPromptCachingMiddleware` 를 사용한다 (`08_integration/11_provider_middleware/06_bedrock_prompt_caching.ipynb`).
두 미들웨어는 파라미터 이름·의미가 거의 같지만 **대상 패키지**와 일부 TTL 제약(Nova는 5m만)이 다르다.

## 체크리스트

- [ ] `usage_metadata.input_token_details`에 `cache_creation` / `cache_read` 가 잡히는지 확인
- [ ] TTL을 `5m`로 둘지 `1h`로 둘지 **대화 세션 길이**로 결정
- [ ] 짧은 대화는 `min_messages_to_cache`로 캐시 생성 자체를 스킵
- [ ] 멀티 프로바이더 파이프라인은 `unsupported_model_behavior="warn"` 명시

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/middleware/anthropic
- Anthropic 문서: Prompt Caching (beta) - ttl 1h 조건
- 다음 노트북: `02_claude_bash_tool.ipynb`